# Build and Train an MNIST Digit Classifier in PyTorch

This notebook walks through building, training, and evaluating a neural network in **PyTorch** to classify handwritten digits from the MNIST dataset.

## 1. Libraries

Import dependencies and setup libraries:

* `import torch`: Core library for tensor math.
* `import torch.nn as nn`: Layers, activations, and loss functions.
* `import torch.optim as optim`: Optimization algorithms (like SGD).
* `from torch.utils.data import DataLoader`: Batches, shuffles, and loads data.
* `from torchvision import datasets, transforms`: Pre-built datasets (like MNIST) and image preprocessing transforms.

In [1]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
# the 2 lines of code above allow us to download a dataset; the semantics are not important so feel free to ignore

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

## 2. Hardware

We route tensors and model parameters to an available accelerator (GPU or Apple Silicon MPS) to speed up calculations:
* **`cuda`**: NVIDIA GPU acceleration interface.
* **`mps`**: Metal Performance Shaders for Apple Silicon chips.
* **`cpu`**: Default system processor if no hardware accelerator is detected.

In [2]:
# Detect hardware accelerator options
device = torch.device(
    "cuda" if torch.cuda.is_available() 
    else "mps" if torch.backends.mps.is_available() 
    else "cpu"
)
print(f"Target hardware device: {device}")

Target hardware device: mps


## 3. Data Pipeline

The MNIST dataset contains $60,000$ training and $10,000$ test images of handwritten digits. Each sample is a $28 \times 28$ grid of grayscale pixel values.

### Image Preprocessing
We use `transforms.Compose` to compile two preprocessing steps:
1. **`transforms.ToTensor()`**: Converts images to PyTorch float tensors, rearranges dimensions to Channel-Height-Width ($C \times H \times W$), and scales pixel values from $[0, 255]$ to $[0.0, 1.0]$.
2. **`transforms.Normalize((0.1307,), (0.3081,))`**: Standardizes pixel values using the MNIST dataset mean (`0.1307`) and standard deviation (`0.3081`). This keeps activations centered, helping gradients stay stable.

### DataLoaders
The dataset is wrapped inside a `DataLoader` to manage iteration:
* **`batch_size=64`**: The model processes images in batches of 64.
* **`shuffle=True`**: Shuffles the training data at the start of each epoch so the model doesn't learn sequence patterns.

In [3]:
# Set up image transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Download datasets if they are not already stored locally in './data'
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Create data loaders to batch and shuffle samples
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Training set batched into: {len(train_loader)} groups of 64 images.")
print(f"Test set batched into:     {len(test_loader)} groups of 64 images.")

100%|██████████| 9.91M/9.91M [00:01<00:00, 8.14MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 897kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.09MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.35MB/s]

Training set batched into: 938 groups of 64 images.
Test set batched into:     157 groups of 64 images.


## 4. Model Architecture

Custom networks inherit from `nn.Module` to track weights and gradients.

### Layers
* **`super().__init__()`**: Initializes the parent module class.
* **`nn.Flatten()`**: Flattens the $1 	imes 28 	imes 28$ image into a $784$ element vector to feed into the linear layer.
* **`nn.Linear(784, 128)`**: Linear layer that projects inputs to 128 features using a weight matrix and bias vector ($y = x W^T + b$).
* **`nn.ReLU()`**: Applies rectified linear activation ($f(x) = \max(0, x)$). This introduces non-linearity, allowing the model to learn complex patterns instead of behaving like a simple linear regression.
* **`nn.Linear(128, 10)`**: Projects 128 features to 10 outputs (one per digit). These raw outputs are called **logits**.

### The `forward` Method
Defines the calculations. Calling `model(x)` runs this method under the hood.

In [4]:
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(784, 128)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(128, 10)
        
    def forward(self, x):
        # x enters as an image tensor of shape (batch_size, 1, 28, 28)
        x = self.flatten(x)   # Reshapes to (batch_size, 784); that is, a list of 1d vectors, where each vector is a flattened image
        x = self.linear1(x)   # Computes linear combination -> (batch_size, 128)
        x = self.relu(x)      # Applies non-linear activation -> (batch_size, 128)
        logits = self.linear2(x) # Computes final logits -> (batch_size, 10)
        return logits

# Instantiate the model and transfer its parameters to the accelerator GPU/CPU memory
model = SimpleMLP().to(device)
print(model)

SimpleMLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear1): Linear(in_features=784, out_features=128, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=128, out_features=10, bias=True)
)


## 5. Loss Function and Optimizer

To train a neural network, we need two key components: a **Loss Function** (to measure how incorrect the model's predictions are) and an **Optimizer** (to adjust the model's weights and reduce the error).

* **Loss Function: `nn.CrossEntropyLoss`**: For multi-class classification, we use Cross-Entropy loss. For detailed concepts and behavior, see [4-loss-funcs.md](4-loss-funcs.md).
* **Optimizer: `optim.SGD`**: Standard Stochastic Gradient Descent is used to update the network parameters based on the gradients. For an explanation of SGD and learning rate selection, see [2-optimizers-schedulers.md](2-optimizers-schedulers.md).

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

## 6. The Training Loop

To train our neural network, we feed the dataset through the model in batches and repeat this process for multiple **epochs**.

For the math behind how gradients are calculated and how parameters are updated, see [1-gradients-backprop.md](1-gradients-backprop.md).

In [6]:
epochs = 5

for epoch in range(epochs):
    model.train() # Set model to training mode at the start of each epoch
    running_loss = 0.0
    
    for batch_idx, (images, targets) in enumerate(train_loader):
        # Transfer image and label tensors to the target hardware device (GPU or CPU)
        images = images.to(device)
        targets = targets.to(device)
        
        # Reset gradient calculations from the previous batch
        optimizer.zero_grad()
        
        # Forward pass: feed inputs through layers to compute logits
        outputs = model(images)
        
        # Compute loss
        loss = criterion(outputs, targets)
        
        # Backward pass: calculate partial derivatives of loss w.r.t parameters
        loss.backward()
        
        # Step parameters using the gradients
        optimizer.step()
        
        # Log average loss status
        running_loss += loss.item() # loss.item() extracts the scalar value from the loss tensor
        if (batch_idx + 1) % 200 == 0:
            avg_batch_loss = running_loss / 200
            print(f"Epoch {epoch + 1}/{epochs} | Batch {batch_idx + 1}/{len(train_loader)} | Loss: {avg_batch_loss:.4f}")
            running_loss = 0.0
            
    print(f"Epoch {epoch + 1}/{epochs} completed.\n")

Epoch 1/5 | Batch 200/938 | Loss: 1.2300
Epoch 1/5 | Batch 400/938 | Loss: 0.5130
Epoch 1/5 | Batch 600/938 | Loss: 0.4246
Epoch 1/5 | Batch 800/938 | Loss: 0.3660
Epoch 1/5 completed.

Epoch 2/5 | Batch 200/938 | Loss: 0.3150
Epoch 2/5 | Batch 400/938 | Loss: 0.3056
Epoch 2/5 | Batch 600/938 | Loss: 0.3010
Epoch 2/5 | Batch 800/938 | Loss: 0.3025
Epoch 2/5 completed.

Epoch 3/5 | Batch 200/938 | Loss: 0.2747
Epoch 3/5 | Batch 400/938 | Loss: 0.2645
Epoch 3/5 | Batch 600/938 | Loss: 0.2546
Epoch 3/5 | Batch 800/938 | Loss: 0.2369
Epoch 3/5 completed.

Epoch 4/5 | Batch 200/938 | Loss: 0.2307
Epoch 4/5 | Batch 400/938 | Loss: 0.2247
Epoch 4/5 | Batch 600/938 | Loss: 0.2335
Epoch 4/5 | Batch 800/938 | Loss: 0.2201
Epoch 4/5 completed.

Epoch 5/5 | Batch 200/938 | Loss: 0.2175
Epoch 5/5 | Batch 400/938 | Loss: 0.1919
Epoch 5/5 | Batch 600/938 | Loss: 0.1985
Epoch 5/5 | Batch 800/938 | Loss: 0.1953
Epoch 5/5 completed.



### Training Step Walkthrough:

Each batch iteration performs the following operations:
1. **Device Transfer (`images.to(device)`)**: Copies data to the GPU or MPS accelerator.
2. **Model Mode (`model.train()`)**: Sets the model to training mode. For how layers like Dropout and BatchNorm behave here, see [3-regularization-normalization.md](3-regularization-normalization.md).
3. **Zeroing Gradients (`optimizer.zero_grad()`)**: Clears out old gradient calculations. For why we reset these buffers, see [1-gradients-backprop.md](1-gradients-backprop.md).
4. **Forward Pass (`model(images)`)**: Computes predictions (logits).
5. **Loss Computation (`criterion(outputs, targets)`)**: Evaluates the error. See [4-loss-funcs.md](4-loss-funcs.md).
6. **Backward Pass (`loss.backward()`)**: Calculates parameter gradients. For a deep dive on backpropagation, see [1-gradients-backprop.md](1-gradients-backprop.md).
7. **Optimizer Step (`optimizer.step()`)**: Updates model weights. For update rules, see [2-optimizers-schedulers.md](2-optimizers-schedulers.md).

## 7. Model Evaluation

Evaluating the model on test data verifies how well it generalizes to unseen samples.

* **`model.eval()`**: Disables training-specific behaviors like dropout.
* **`with torch.no_grad()`**: Tells PyTorch not to track gradients. This saves memory and speeds up computation during evaluation.
* **`outputs.max(dim=1)`**: Gets the index of the highest prediction value (logit) along the class dimension, which represents the predicted digit.

In [7]:
model.eval() # Activate evaluation behavior state
correct = 0
total = 0

with torch.no_grad(): # Disable gradient calculation to save memory
    for images, targets in test_loader:
        images = images.to(device)
        targets = targets.to(device)
        
        # Compute logits
        outputs = model(images)
        
        # Get predicted labels by finding the index with the highest logit output
        # max(dim=1) returns values and indexes; we only want the indexes (predicted)
        _, predicted = outputs.max(dim=1)
        
        # Accumulate correct sample counts
        total += targets.size(0)
        correct += (predicted == targets).sum().item()

accuracy = 100.0 * correct / total
print(f"Test Set Evaluation:")
print(f"  Correct Predictions: {correct} / {total}")
print(f"  Accuracy Score:      {accuracy:.2f}%")

Test Set Evaluation:
  Correct Predictions: 9465 / 10000
  Accuracy Score:      94.65%


94.65% accuracy! Yay!